# DEH 30-Day PySpark Challenge
## Day 8 — Creating DataFrames, Unique Records & Dropping Duplicates

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — createDataFrame() with list of tuples

In [ ]:
# Method 1 — list of tuples + column names
data = [
    ("O0001", "C001", 1299.99, "Delivered"),
    ("O0002", "C002",  449.99, "Delivered"),
    ("O0003", "C003",  349.99, "Shipped")
]

df = spark.createDataFrame(data, ["order_id", "customer_id", "unit_price", "status"])
df.show()
df.printSchema()

## Step 5 — createDataFrame() with explicit schema

In [ ]:
# Method 2 — explicit schema
schema = StructType([
    StructField("order_id",    StringType(),  False),
    StructField("customer_id", StringType(),  True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("quantity",    IntegerType(), True)
])

data = [
    ("O0001", "C001", 1299.99, 2),
    ("O0002", "C002",  449.99, 1),
    ("O0003", "C003",  349.99, 4)
]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

## Step 6 — createDataFrame() with Row objects

In [ ]:
# Method 3 — From a CSV file
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Type : {type(orders_df)}")
print(f"Rows : {orders_df.count()}")
orders_df.show(3)

## Step 7 — Handling None values

In [ ]:
# None in Python = null in Spark
data_with_nulls = [
    ("O0001", "C001", 1299.99),
    ("O0002", None,    449.99),
    ("O0003", "C003",  None)
]

df_nulls = spark.createDataFrame(
    data_with_nulls,
    ["order_id", "customer_id", "unit_price"]
)
df_nulls.show()

## Step 7 — Method 4: From an RDD

In [ ]:
# Method 4 — From an RDD
sc = spark.sparkContext

# Create an RDD of tuples
rdd = sc.parallelize([
    ("O0001", "C001", 1299.99, "Delivered"),
    ("O0002", "C002",  449.99, "Delivered"),
    ("O0003", "C003",  349.99, "Shipped")
])

# Convert to DataFrame using toDF()
df_from_rdd = rdd.toDF(["order_id", "customer_id", "unit_price", "status"])
df_from_rdd.show()
df_from_rdd.printSchema()

# Or use createDataFrame() with explicit schema for guaranteed types
schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("unit_price",  DoubleType(), True),
    StructField("status",      StringType(), True)
])

df_from_rdd2 = spark.createDataFrame(rdd, schema)
df_from_rdd2.printSchema()

## Step 8 — distinct() — unique values

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

# Unique regions
print("Unique regions:")
orders_df.select("region").distinct().show()

# Unique statuses
print("Unique statuses:")
orders_df.select("status").distinct().show()

print(f"Unique regions  : {orders_df.select('region').distinct().count()}")
print(f"Unique statuses : {orders_df.select('status').distinct().count()}")
print(f"Unique customers: {orders_df.select('customer_id').distinct().count()}")

In [ ]:
# Unique combinations of multiple columns
orders_df.select("region", "status").distinct().orderBy("region", "status").show()

## Step 9 — dropDuplicates()

In [ ]:
# Create DataFrame with duplicate rows
dupes_data = [
    ("O0001", "C001", 1299.99, "Delivered"),
    ("O0002", "C002",  449.99, "Delivered"),
    ("O0001", "C001", 1299.99, "Delivered"),  # duplicate
    ("O0003", "C003",  349.99, "Shipped"),
    ("O0002", "C002",  449.99, "Delivered")   # duplicate
]

df_dupes = spark.createDataFrame(
    dupes_data,
    ["order_id", "customer_id", "unit_price", "status"]
)

print(f"Before: {df_dupes.count()} rows")
df_clean = df_dupes.dropDuplicates()
print(f"After : {df_clean.count()} rows")
df_clean.show()

---
## Assignment — Day 8

---

### Task 1
Create a DataFrame from scratch using `createDataFrame()` with an explicit schema.  
Use 5 rows with columns: `employee_id` (String), `name` (String), `department` (String), `salary` (Double).  
Include at least one `None` value. Show the DataFrame and print the schema.

In [ ]:
# Task 1 — Write your code here


### Task 2
From `orders.csv`, find all unique values of `payment_method` using `distinct()`.  
Then find all unique combinations of `region` + `status`.  
How many unique combinations are there?

In [ ]:
# Task 2 — Write your code here


### Task 3
Create a DataFrame with intentional duplicates — repeat at least 2 rows exactly.  
Use `dropDuplicates()` with no arguments to remove exact duplicates.  
Print the row count before and after.

In [ ]:
# Task 3 — Write your code here


### Task 4
From `orders.csv`, use `dropDuplicates(["customer_id"])` to keep only the first order per customer.  
How many rows remain? What does this tell you about the dataset?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*